# Strategy 2 — Director / Actor Affinity

Strategy 2 walks two hops: from a user, through their 4★+ ratings, to the **directors** and **top-billed actors** of those movies, then *back out* to other movies those people made — movies the user hasn't rated. The score adds up *how strongly the user likes each person* across every connecting top person. The story for one rec is a chain — *user → seed movie → person → rec* — which is exactly what notebook 11 deliberately deferred.

A few notes on what this notebook is and isn't:

- **OR-joined, not AND.** A movie can enter the candidate pool because it shares a top director *or* a top actor — intersecting the two routes would shrink the pool to almost nothing.
- **`billing_order <= 5` filter on actors.** Walk-on parts pollute the affinity signal; the filter keeps actor evidence to above-the-title cast.
- **Per-rec single-chain explainability.** Each rec carries `via_people` (the connecting top people) and up to 3 `seed_chains` of the form *"<seed> → <person> (<kind>) → <rec>"*. The chain *is* the explanation.
- **Anchor user: 196.** Same recurring anchor as notebooks 09 / 10 / 11.
- **Notebook structure: build the query phase by phase first.** Cells 4–6 run Phase 1 / Phase 2 / Phase 3 as standalone Cypher with their own `RETURN`. Cells 7–9 then reveal the unified `DIRECTOR_ACTOR_CYPHER` constant and confirm the wrapper function returns the same rows.

In [15]:
# 1. Setup
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from graph import run_query, close_driver

ANCHOR_USER = 196
print(f"Anchor user: {ANCHOR_USER}")

Anchor user: 196


In [16]:
# 2. User 196's people profile — what we expect Strategy 2 to surface
top_dirs = run_query(
    """
    MATCH (u:User {userId: $userId})-[r:RATED]->(m:Movie)-[:DIRECTED_BY]->(d:Director)
    WHERE r.rating >= 4.0
    RETURN d.name AS director, count(m) AS affinity
    ORDER BY affinity DESC
    LIMIT 3
    """,
    {"userId": ANCHOR_USER},
)
print("Top 3 directors by 4★+ count:")
print(pd.DataFrame(top_dirs))

top_actors = run_query(
    """
    MATCH (u:User {userId: $userId})-[r:RATED]->(m:Movie)-[fa:FEATURES_ACTOR]->(a:Actor)
    WHERE r.rating >= 4.0 AND fa.billing_order <= 5
    RETURN a.name AS actor, count(m) AS affinity
    ORDER BY affinity DESC
    LIMIT 5
    """,
    {"userId": ANCHOR_USER},
)
print("\nTop 5 top-billed actors (billing_order <= 5) by 4★+ count:")
print(pd.DataFrame(top_actors))

Top 3 directors by 4★+ count:
      director  affinity
0    Hal Ashby         2
1  Tom Shadyac         2
2   Rob Reiner         2

Top 5 top-billed actors (billing_order <= 5) by 4★+ count:
              actor  affinity
0       Bill Hunter         2
1      Hugo Weaving         2
2    Robin Williams         2
3  Miriam Margolyes         1
4      Miriam Flynn         1


## Let's build the query phase by phase

Before showing the unified query (cell 8) and calling the wrapper function (cell 9), let's run each phase as its own Cypher block:

- **Phase 1 — top people.** From the user's ≥4★ ratings, find their top 3 directors and top 5 top-billed actors. Output: a list of `{name, kind, affinity, seeds}` rows.
- **Phase 2 — seed evidence.** For each top person, surface the user's actual ≥4★ ratings that earned them their affinity. Output: one row per (seed, person) bridge.
- **Phase 3 — forward walk.** For each top person, find unseen recs they're connected to (via `:DIRECTED_BY` or `:FEATURES_ACTOR` with the billing filter), aggregate per rec, score, rank.

Each cell below ends with `RETURN` so it runs standalone. In the unified version (cell 8), those `RETURN`s become `WITH` — same callback to the `WITH` ↔ `RETURN` distinction from notebook 09's Cypher 101 walkthrough.

In [17]:
# 4. Phase 1 standalone — top 3 directors + top 5 top-billed actors as a unified people list
PHASE_1 = """
MATCH (u:User {userId: $userId})-[r1:RATED]->(seed_d:Movie)-[:DIRECTED_BY]->(d:Director)
WHERE r1.rating >= $threshold
WITH u, d, count(seed_d) AS affinity, collect(DISTINCT seed_d.title) AS seed_titles
ORDER BY affinity DESC
LIMIT $top_n_directors
WITH u, collect({
  person_id: elementId(d), name: d.name, kind: 'director',
  affinity: affinity, seeds: seed_titles
}) AS top_dirs

MATCH (u)-[r2:RATED]->(seed_a:Movie)-[fa:FEATURES_ACTOR]->(a:Actor)
WHERE r2.rating >= $threshold AND fa.billing_order <= $max_billing
WITH u, top_dirs, a, count(seed_a) AS affinity, collect(DISTINCT seed_a.title) AS seed_titles
ORDER BY affinity DESC
LIMIT $top_n_actors
WITH u, top_dirs, collect({
  person_id: elementId(a), name: a.name, kind: 'actor',
  affinity: affinity, seeds: seed_titles
}) AS top_actors
WITH u, top_dirs + top_actors AS top_people

UNWIND top_people AS p
RETURN p.name AS name, p.kind AS kind, p.affinity AS affinity, size(p.seeds) AS n_seeds
ORDER BY kind DESC, affinity DESC
"""

phase_1_rows = run_query(
    PHASE_1,
    {"userId": ANCHOR_USER, "threshold": 4.0, "top_n_directors": 3,
     "top_n_actors": 5, "max_billing": 5},
)
print("Phase 1 — top_people:")
print(pd.DataFrame(phase_1_rows))

Phase 1 — top_people:
               name      kind  affinity  n_seeds
0         Hal Ashby  director         2        2
1       Tom Shadyac  director         2        2
2        Rob Reiner  director         2        2
3       Bill Hunter     actor         2        2
4      Hugo Weaving     actor         2        2
5    Robin Williams     actor         2        2
6  Miriam Margolyes     actor         1        1
7      Miriam Flynn     actor         1        1


In [18]:
# 5. Phase 2 standalone — seed evidence per person (which >=4* ratings earned each person their affinity)
PHASE_2 = """
MATCH (u:User {userId: $userId})-[r1:RATED]->(seed_d:Movie)-[:DIRECTED_BY]->(d:Director)
WHERE r1.rating >= $threshold
WITH u, d, count(seed_d) AS affinity, collect(DISTINCT seed_d.title) AS seed_titles
ORDER BY affinity DESC
LIMIT $top_n_directors
WITH u, collect({
  person_id: elementId(d), name: d.name, kind: 'director',
  affinity: affinity, seeds: seed_titles
}) AS top_dirs

MATCH (u)-[r2:RATED]->(seed_a:Movie)-[fa:FEATURES_ACTOR]->(a:Actor)
WHERE r2.rating >= $threshold AND fa.billing_order <= $max_billing
WITH u, top_dirs, a, count(seed_a) AS affinity, collect(DISTINCT seed_a.title) AS seed_titles
ORDER BY affinity DESC
LIMIT $top_n_actors
WITH u, top_dirs, collect({
  person_id: elementId(a), name: a.name, kind: 'actor',
  affinity: affinity, seeds: seed_titles
}) AS top_actors
WITH u, top_dirs + top_actors AS top_people

UNWIND top_people AS p
UNWIND p.seeds AS seed
RETURN seed AS seed_title, p.name AS person, p.kind AS kind, p.affinity AS affinity
ORDER BY kind DESC, affinity DESC, seed
"""

phase_2_rows = run_query(
    PHASE_2,
    {"userId": ANCHOR_USER, "threshold": 4.0, "top_n_directors": 3,
     "top_n_actors": 5, "max_billing": 5},
)
print(f"Phase 2 — {len(phase_2_rows)} seed→person bridges. First 15:")
display(pd.DataFrame(phase_2_rows).head(15))

Phase 2 — 14 seed→person bridges. First 15:


,seed_title,person,kind,affinity
0,Ace Ventura: Pet Detective,Tom Shadyac,director,2
1,"American President, The",Rob Reiner,director,2
2,Being There,Hal Ashby,director,2
3,Harold and Maude,Hal Ashby,director,2
4,"Nutty Professor, The",Tom Shadyac,director,2
5,Stand by Me,Rob Reiner,director,2
6,"Adventures of Priscilla, Queen of the Desert, The",Bill Hunter,actor,2
7,"Adventures of Priscilla, Queen of the Desert, The",Hugo Weaving,actor,2
8,Babe,Hugo Weaving,actor,2
9,"Birdcage, The",Robin Williams,actor,2


In [19]:
# 6. Phase 3 standalone — walk forward to unseen recs, score, rank
PHASE_3 = """
MATCH (u:User {userId: $userId})-[r1:RATED]->(seed_d:Movie)-[:DIRECTED_BY]->(d:Director)
WHERE r1.rating >= $threshold
WITH u, d, count(seed_d) AS affinity, collect(DISTINCT seed_d.title) AS seed_titles
ORDER BY affinity DESC
LIMIT $top_n_directors
WITH u, collect({
  person_id: elementId(d), name: d.name, kind: 'director',
  affinity: affinity, seeds: seed_titles
}) AS top_dirs

MATCH (u)-[r2:RATED]->(seed_a:Movie)-[fa:FEATURES_ACTOR]->(a:Actor)
WHERE r2.rating >= $threshold AND fa.billing_order <= $max_billing
WITH u, top_dirs, a, count(seed_a) AS affinity, collect(DISTINCT seed_a.title) AS seed_titles
ORDER BY affinity DESC
LIMIT $top_n_actors
WITH u, top_dirs, collect({
  person_id: elementId(a), name: a.name, kind: 'actor',
  affinity: affinity, seeds: seed_titles
}) AS top_actors
WITH u, top_dirs + top_actors AS top_people

UNWIND top_people AS tp
MATCH (rec:Movie)
WHERE NOT EXISTS { (u)-[:RATED]->(rec) }
  AND (
    (tp.kind = 'director' AND EXISTS {
      MATCH (rec)-[:DIRECTED_BY]->(d2:Director) WHERE elementId(d2) = tp.person_id
    })
    OR
    (tp.kind = 'actor' AND EXISTS {
      MATCH (rec)-[fa2:FEATURES_ACTOR]->(a2:Actor)
      WHERE elementId(a2) = tp.person_id AND fa2.billing_order <= $max_billing
    })
  )
WITH rec, collect(tp) AS connecting_people
RETURN rec.title AS title,
       rec.year  AS year,
       reduce(s = 0, p IN connecting_people | s + p.affinity) AS score,
       [p IN connecting_people | p.name + ' (' + p.kind + ')'] AS via_people,
       [p IN connecting_people | head(p.seeds) + ' → ' + p.name + ' (' + p.kind + ') → ' + rec.title][..3] AS seed_chains
ORDER BY score DESC, title ASC
LIMIT $k
"""

phase_3_rows = run_query(
    PHASE_3,
    {"userId": ANCHOR_USER, "threshold": 4.0, "top_n_directors": 3,
     "top_n_actors": 5, "max_billing": 5, "k": 10},
)

def render(rows):
    if not rows:
        return pd.DataFrame(columns=["title", "year", "score", "via_people", "seed_chains"])
    df = pd.DataFrame(rows)
    df["via_people"] = df["via_people"].apply(lambda xs: ", ".join(xs))
    df["seed_chains"] = df["seed_chains"].apply(lambda xs: "\n".join(xs))
    return df[["title", "year", "score", "via_people", "seed_chains"]]

render(phase_3_rows)

,title,year,score,via_people,seed_chains
0,Aladdin,1992.0,2,Robin Williams (actor),"Birdcage, The → Robin Williams (actor) → Aladdin"
1,Aladdin and the King of Thieves,1996.0,2,Robin Williams (actor),"Birdcage, The → Robin Williams (actor) → Aladd..."
2,Being Human,1993.0,2,Robin Williams (actor),"Birdcage, The → Robin Williams (actor) → Being..."
3,Dead Poets Society,1989.0,2,Robin Williams (actor),"Birdcage, The → Robin Williams (actor) → Dead ..."
4,Fathers' Day,1997.0,2,Robin Williams (actor),"Birdcage, The → Robin Williams (actor) → Fathe..."
5,Flubber,1997.0,2,Robin Williams (actor),"Birdcage, The → Robin Williams (actor) → Flubber"
6,Ghosts of Mississippi,1996.0,2,Rob Reiner (director),Stand by Me → Rob Reiner (director) → Ghosts o...
7,Good Will Hunting,1997.0,2,Robin Williams (actor),"Birdcage, The → Robin Williams (actor) → Good ..."
8,Jack,1996.0,2,Robin Williams (actor),"Birdcage, The → Robin Williams (actor) → Jack"
9,Jumanji,1995.0,2,Robin Williams (actor),"Birdcage, The → Robin Williams (actor) → Jumanji"


## Now bring it all together

Phases 1–3 above are the body of the unified `DIRECTOR_ACTOR_CYPHER` constant in [`src/recommender.py`](../src/recommender.py). The collapse is mechanical:

- Each standalone phase ended with `RETURN` so it could run on its own.
- In the unified query, those `RETURN`s become `WITH` — the same variables flow into the next phase instead of leaving the query.
- The final `RETURN` lives at the end of Phase 3, producing the rec list.

Cell 8 prints the constant. Cell 9 calls the wrapper function and confirms the output matches Phase 3 above row-for-row.

In [20]:
# 8. The unified query — same Cypher you just ran phase by phase, chained with WITH
from recommender import DIRECTOR_ACTOR_CYPHER
print(DIRECTOR_ACTOR_CYPHER)


// Phase 1a — user's top directors by count of >= threshold ratings
MATCH (u:User {userId: $userId})-[r1:RATED]->(seed_d:Movie)-[:DIRECTED_BY]->(d:Director)
WHERE r1.rating >= $threshold
WITH u, d, count(seed_d) AS affinity, collect(DISTINCT seed_d.title) AS seed_titles
ORDER BY affinity DESC
LIMIT $top_n_directors
WITH u, collect({
  person_id: elementId(d),
  name: d.name,
  kind: 'director',
  affinity: affinity,
  seeds: seed_titles
}) AS top_dirs

// Phase 1b — user's top top-billed actors
MATCH (u)-[r2:RATED]->(seed_a:Movie)-[fa:FEATURES_ACTOR]->(a:Actor)
WHERE r2.rating >= $threshold AND fa.billing_order <= $max_billing
WITH u, top_dirs, a, count(seed_a) AS affinity, collect(DISTINCT seed_a.title) AS seed_titles
ORDER BY affinity DESC
LIMIT $top_n_actors
WITH u, top_dirs, collect({
  person_id: elementId(a),
  name: a.name,
  kind: 'actor',
  affinity: affinity,
  seeds: seed_titles
}) AS top_actors
WITH u, top_dirs + top_actors AS top_people

// Phase 3 — walk forward to unsee

In [21]:
# 9. Call the wrapper — should match Phase 3 cell 6 row-for-row
from recommender import director_actor_affinity
recs = director_actor_affinity(ANCHOR_USER)
render(recs)

,title,year,score,via_people,seed_chains
0,Aladdin,1992.0,2,Robin Williams (actor),"Birdcage, The → Robin Williams (actor) → Aladdin"
1,Aladdin and the King of Thieves,1996.0,2,Robin Williams (actor),"Birdcage, The → Robin Williams (actor) → Aladd..."
2,Being Human,1993.0,2,Robin Williams (actor),"Birdcage, The → Robin Williams (actor) → Being..."
3,Dead Poets Society,1989.0,2,Robin Williams (actor),"Birdcage, The → Robin Williams (actor) → Dead ..."
4,Fathers' Day,1997.0,2,Robin Williams (actor),"Birdcage, The → Robin Williams (actor) → Fathe..."
5,Flubber,1997.0,2,Robin Williams (actor),"Birdcage, The → Robin Williams (actor) → Flubber"
6,Ghosts of Mississippi,1996.0,2,Rob Reiner (director),Stand by Me → Rob Reiner (director) → Ghosts o...
7,Good Will Hunting,1997.0,2,Robin Williams (actor),"Birdcage, The → Robin Williams (actor) → Good ..."
8,Jack,1996.0,2,Robin Williams (actor),"Birdcage, The → Robin Williams (actor) → Jack"
9,Jumanji,1995.0,2,Robin Williams (actor),"Birdcage, The → Robin Williams (actor) → Jumanji"


## Reading one explainability chain

Pick the top rec from cell 9. Read its `via_people` — that's the list of top directors and actors connecting user 196 to the rec. Then read one of its `seed_chains` end-to-end:

> *"User 196 rated **<seed>** 4★+ → that movie was directed by / featured **<person>** → **<rec>** also features **<person>** → here's the recommendation."*

The chain *is* the explanation. Notebook 11's Strategy 1 deliberately did not produce per-rec chains because Strategy 1's `score` is an aggregate over genres — pretending the chain `seed → genre → rec` is causal would mislead. Strategy 2 is different: the structure of the query *is* a chain, so the explanation honestly mirrors it.

Two more things to internalize:

1. **`score` is a ranking signal, not a confidence percentage.** A score of 12 means *"sum of affinities across matched top people = 12"* — useful for ordering, not for probability claims.
2. **`seed_chains` is capped at 3 in unwind order**, not strength-ranked. Director chains appear first (top directors are unwound before top actors), then actor chains. All chains in the list are real; none is *"the strongest."*

In [22]:
# 11. Knob — top_n_directors / top_n_actors
narrow = director_actor_affinity(ANCHOR_USER, top_n_directors=1, top_n_actors=1)
wide = director_actor_affinity(ANCHOR_USER, top_n_directors=5, top_n_actors=10)

print("(top_n_directors=1, top_n_actors=1) — tightest per-rec story:")
print(render(narrow).head(3))
print("\n(top_n_directors=5, top_n_actors=10) — wider candidate pool, bigger scores from multi-person overlap:")
print(render(wide).head(3))

(top_n_directors=1, top_n_actors=1) — tightest per-rec story:
       title    year  score              via_people  \
0  Liar Liar  1997.0      2  Tom Shadyac (director)   

                                         seed_chains  
0  Ace Ventura: Pet Detective → Tom Shadyac (dire...  

(top_n_directors=5, top_n_actors=10) — wider candidate pool, bigger scores from multi-person overlap:
                             title    year  score              via_people  \
0                          Aladdin  1992.0      2  Robin Williams (actor)   
1  Aladdin and the King of Thieves  1996.0      2  Robin Williams (actor)   
2                      Being Human  1993.0      2  Robin Williams (actor)   

                                         seed_chains  
0   Birdcage, The → Robin Williams (actor) → Aladdin  
1  Birdcage, The → Robin Williams (actor) → Aladd...  
2  Birdcage, The → Robin Williams (actor) → Being...  


In [23]:
# 12. Knob — max_billing (turn the actor filter off)
no_filter = director_actor_affinity(ANCHOR_USER, max_billing=999)
print("max_billing=999 — extras and walk-ons now count toward affinity. Top recs:")
print(render(no_filter).head(5))
print("\nCompare with cell 9's default (max_billing=5). With the filter off, lower-quality recs")
print("surface because uncredited or single-line actors count just as much as leads do.")

max_billing=999 — extras and walk-ons now count toward affinity. Top recs:
                             title    year  score  \
0                        Liar Liar  1997.0      4   
1                          Aladdin  1992.0      2   
2  Aladdin and the King of Thieves  1996.0      2   
3                      Being Human  1993.0      2   
4               Dead Poets Society  1989.0      2   

                                   via_people  \
0  Tom Shadyac (director), Anne Haney (actor)   
1                      Robin Williams (actor)   
2                      Robin Williams (actor)   
3                      Robin Williams (actor)   
4                      Robin Williams (actor)   

                                         seed_chains  
0  Ace Ventura: Pet Detective → Tom Shadyac (dire...  
1   Birdcage, The → Robin Williams (actor) → Aladdin  
2  Birdcage, The → Robin Williams (actor) → Aladd...  
3  Birdcage, The → Robin Williams (actor) → Being...  
4  Birdcage, The → Robin Williams (a

In [24]:
# 13. Edge case — cold start
# 13a. A user with zero >=4* ratings, if one exists in the dataset
no_signal_user = run_query(
    """
    MATCH (u:User)
    WHERE NOT EXISTS { MATCH (u)-[r:RATED]->() WHERE r.rating >= 4 }
    RETURN u.userId AS userId LIMIT 1
    """,
)
if no_signal_user:
    cold_id = no_signal_user[0]["userId"]
    n = len(director_actor_affinity(cold_id))
    print(f"13a. User {cold_id} has zero >=4* ratings -> director_actor_affinity returned {n} recs.")
else:
    print("13a. (No >=4*-free user in MovieLens 100K — skipping.)")

# 13b. Non-existent user
n_ghost = len(director_actor_affinity(99999))
print(f"13b. User 99999 doesn't exist -> director_actor_affinity returned {n_ghost} recs.")

print("\nContract: empty list = no signal. Same as Strategy 1; Strategy 4 owns the fallback.")

13a. User 685 has zero >=4* ratings -> director_actor_affinity returned 0 recs.
13b. User 99999 doesn't exist -> director_actor_affinity returned 0 recs.

Contract: empty list = no signal. Same as Strategy 1; Strategy 4 owns the fallback.


In [25]:
# 14. Anchor sanity — none of the recs should already be rated by user 196
rec_ids = [r["movieId"] for r in recs]
overlap = run_query(
    """
    MATCH (u:User {userId: $userId})-[:RATED]->(m:Movie)
    WHERE m.movieId IN $ids
    RETURN count(m) AS overlap
    """,
    {"userId": ANCHOR_USER, "ids": rec_ids},
)
n = overlap[0]["overlap"] if overlap else -1
print("OK — 0 overlap" if n == 0 else f"MISMATCH — user 196 has already rated {n} of these recs")

OK — 0 overlap


In [26]:
# 15. Cleanup
close_driver()
print("Driver closed.")

Driver closed.
